In [1]:
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import pandas as pd
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
import pickle
import numpy as np
import ast
from time import time
from scipy.spatial import distance
import math
from tqdm import tqdm
import joblib

BASE_DIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"

# remote 
NEO4J_URL ="bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PWD = "password"

# local
#NEO4J_URL ="bolt://localhost:7687"
#NEO4J_USER = "neo4j"
#NEO4J_PWD = "neo4j"

import sys 
sys.path.insert(0, "../")

from src.neo4j_functions import Neo4jConnection
from src.utils.data_structs import NodeCreator, NODES_TYPES_MAP, create_id_for_node_pair
from src.embedding_functions import ChromaConnection, VectorDBConnectionConfig
from src.qa_pipeline.knowledge_retriever.astar_utils import AerospikeConnector, KVDBConnectionConfig

COLLECTION_NAME = 'diaasq2'
NODES_DB_PATH = '../data/graph_structures/vectorized_nodes/v10/densedb'

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Calculating distances between nodes

In [2]:
distances = {
    'l2': distance.euclidean,
    'ip': lambda v1, v2: 1-np.dot(v1, v2)
}

In [3]:
# подключиться к коллекции с нодами
nodes_db = ChromaConnection(
    VectorDBConnectionConfig(path=NODES_DB_PATH, db_name="vectorized_nodes")
)
print(nodes_db.collection.count())

71338


In [4]:
# подключиться к бд с ключами
kv_db = AerospikeConnector()

In [5]:
data = nodes_db.collection.get(include=['embeddings'])

In [ ]:
n12_ids, n12_dists = [], []

for i in tqdm(range(len(data['ids'])-1)):
    n1_id = data['ids'][i]
    n1_emb = data['embeddings'][i]
    for j in range(i+1, len(data['ids'])):
        n2_id = data['ids'][j]
        n2_emb = data['embeddings'][j]
        
        n12_dists.append(distances['ip'](n1_emb, n2_emb))
        n12_ids.append(create_id_for_node_pair(n1_id, n2_id))

In [10]:
kv_db.create(
    list(map(lambda id: ('test', 'diaasq2', id), n12_ids)), 
    list(map(lambda val: {'dist': val}))
)

'3e67b553f0f67802f62c33dc200d97c7'